# To build a decision tree from scratch and implement it in breast cancer classification

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

In [ ]:
df = pd.read_csv('/data.csv')

In [4]:
print("Data Types and Missing Values:")
print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Data Types and Missing Values:

Data Types:
id                           int64
diagnosis                   object
radius_mean                float64
texture_mean               float64
perimeter_mean             float64
area_mean                  float64
smoothness_mean            float64
compactness_mean           float64
concavity_mean             float64
concave points_mean        float64
symmetry_mean              float64
fractal_dimension_mean     float64
radius_se                  float64
texture_se                 float64
perimeter_se               float64
area_se                    float64
smoothness_se              float64
compactness_se             float64
concavity_se               float64
concave points_se          float64
symmetry_se                float64
fractal_dimension_se       float64
radius_worst               float64
texture_worst              float64
perimeter_worst            float64
area_worst                 float64
smoothness_worst           float64
compactness

In [5]:
cols_to_drop = ['id','Unnamed: 32']
for col in cols_to_drop:
    df = df.drop(columns=[col])
    print(f"Dropped {col}")

Dropped id
Dropped Unnamed: 32


In [19]:
df['target'] = df['diagnosis'].map({'M': 1, 'B': 0})
target_col = 'target'

print(f"\nTarget column: {target_col}")
print(f"Unique values: {df[target_col].unique()}")
print(f"\nClass distribution:")
print(df[target_col].value_counts())


Target column: target
Unique values: [1 0]

Class distribution:
target
0    357
1    212
Name: count, dtype: int64


In [30]:
class SimpleDecisionTree:
    """A decision tree for classification (works with any numerical dataset)"""
    
    def __init__(self, max_depth=5, min_samples_split=2, min_impurity_decrease=0.0):
        """
        Parameters:
        - max_depth: Maximum depth of the tree
        - min_samples_split: Minimum samples required to split a node
        - min_impurity_decrease: Minimum gain required to split
        """
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_impurity_decrease = min_impurity_decrease
        self.tree = None  # Will store the tree structure
        
    def impurity(self, y):
        """Calculate impurity (1 - max class probability)"""
        if len(y) == 0:
            return 0
        # Using majority class impurity (simple version)
        most_common = np.bincount(y.astype(int)).max()
        return 1 - (most_common / len(y))
    
    def gini_impurity(self, y):
        """Gini impurity (more common in decision trees)"""
        if len(y) == 0:
            return 0
        classes = np.unique(y)
        prob = [np.sum(y == c) / len(y) for c in classes]
        return 1 - sum(p**2 for p in prob)
    
    def entropy(self, y):
        """Entropy (alternative impurity measure)"""
        if len(y) == 0:
            return 0
        classes = np.unique(y)
        prob = [np.sum(y == c) / len(y) for c in classes]
        return -sum(p * np.log2(p) for p in prob if p > 0)
    
    def information_gain(self, y_parent, y_left, y_right, method='gini'):
        """Calculate information gain from split"""
        if method == 'gini':
            parent_impurity = self.gini_impurity(y_parent)
            left_impurity = self.gini_impurity(y_left)
            right_impurity = self.gini_impurity(y_right)
        elif method == 'entropy':
            parent_impurity = self.entropy(y_parent)
            left_impurity = self.entropy(y_left)
            right_impurity = self.entropy(y_right)
        else:
            parent_impurity = self.impurity(y_parent)
            left_impurity = self.impurity(y_left)
            right_impurity = self.impurity(y_right)
        
        n_parent = len(y_parent)
        n_left = len(y_left)
        n_right = len(y_right)
        
        weighted_impurity = (n_left/n_parent)*left_impurity + (n_right/n_parent)*right_impurity
        
        return parent_impurity - weighted_impurity
    
    def find_best_split(self, X, y):
        """Find the best split for the current node"""
        best_gain = 0
        best_split = None
        n_samples, n_features = X.shape
        
        # Check if we should split
        if len(y) < self.min_samples_split:
            return None
        
        # Try each feature
        for feature_idx in range(n_features):
            # Get unique values for this feature
            thresholds = np.unique(X[:, feature_idx])
            
            # Try each threshold (for efficiency, sample if too many)
            if len(thresholds) > 100:
                thresholds = np.percentile(X[:, feature_idx], np.linspace(0, 100, 50))
            
            for threshold in thresholds:
                # Split the data
                left_mask = X[:, feature_idx] < threshold
                right_mask = ~left_mask
                
                # Skip if split is invalid
                if len(y[left_mask]) == 0 or len(y[right_mask]) == 0:
                    continue
                
                # Calculate gain
                gain = self.information_gain(y, y[left_mask], y[right_mask])
                
                # Update best split
                if gain > best_gain:
                    best_gain = gain
                    best_split = {
                        'feature_idx': feature_idx,
                        'threshold': threshold,
                        'gain': gain,
                        'left_mask': left_mask,
                        'right_mask': right_mask
                    }
        
        # Only return split if gain is significant
        if best_split and best_split['gain'] > self.min_impurity_decrease:
            return best_split
        return None
    
    def build_tree(self, X, y, depth=0):
        """Recursively build the decision tree"""
        n_samples, n_features = X.shape
        n_classes = len(np.unique(y))
        
        # Stopping conditions
        if (depth >= self.max_depth or 
            n_classes == 1 or 
            n_samples < self.min_samples_split):
            # Return leaf node with majority class
            return {
                'type': 'leaf',
                'prediction': np.bincount(y.astype(int)).argmax(),
                'n_samples': n_samples,
                'class_distribution': dict(Counter(y))
            }
        
        # Find best split
        best_split = self.find_best_split(X, y)
        
        if best_split is None:
            # No good split found, make leaf
            return {
                'type': 'leaf',
                'prediction': np.bincount(y.astype(int)).argmax(),
                'n_samples': n_samples,
                'class_distribution': dict(Counter(y))
            }
        
        # Split the data
        left_mask = best_split['left_mask']
        right_mask = best_split['right_mask']
        
        X_left, y_left = X[left_mask], y[left_mask]
        X_right, y_right = X[right_mask], y[right_mask]
        
        # Recursively build left and right subtrees
        return {
            'type': 'split',
            'feature_idx': best_split['feature_idx'],
            'threshold': best_split['threshold'],
            'gain': best_split['gain'],
            'left': self.build_tree(X_left, y_left, depth + 1),
            'right': self.build_tree(X_right, y_right, depth + 1),
            'n_samples': n_samples,
            'class_distribution': dict(Counter(y))
        }
    
    def fit(self, X, y):
        """Train the decision tree"""
        # Convert to numpy arrays if they're DataFrames/Series
        if hasattr(X, 'values'):
            X = X.values
        if hasattr(y, 'values'):
            y = y.values
        
        # Ensure y is integer type
        y = y.astype(int)
        
        # Build the tree
        self.tree = self.build_tree(X, y)
        self.feature_names = None  # Will be set later if provided
        
        return self
    
    def predict_single(self, x, node):
        """Predict for a single sample"""
        if node['type'] == 'leaf':
            return node['prediction']
        else:
            if x[node['feature_idx']] < node['threshold']:
                return self.predict_single(x, node['left'])
            else:
                return self.predict_single(x, node['right'])
    
    def predict(self, X):
        """Make predictions for multiple samples"""
        if hasattr(X, 'values'):
            X = X.values
        
        predictions = []
        for sample in X:
            predictions.append(self.predict_single(sample, self.tree))
        
        return np.array(predictions)
    
    def print_tree(self, node=None, depth=0, feature_names=None):
        """Print the decision tree structure"""
        if node is None:
            node = self.tree
            if feature_names is None:
                feature_names = [f"Feature_{i}" for i in range(30)]
            self.feature_names = feature_names
        
        if node['type'] == 'leaf':
            indent = "  " * depth
            print(f"{indent}└── Predict: Class {node['prediction']} (samples: {node['n_samples']}, distribution: {node['class_distribution']})")
        else:
            indent = "  " * depth
            feature_name = feature_names[node['feature_idx']] if feature_names else f"Feature_{node['feature_idx']}"
            print(f"{indent}├── If {feature_name} < {node['threshold']:.4f} (gain: {node['gain']:.4f}, samples: {node['n_samples']})")
            print(f"{indent}│   └── Then:")
            self.print_tree(node['left'], depth + 2, feature_names)
            print(f"{indent}└── Else:")
            self.print_tree(node['right'], depth + 2, feature_names)
    
    def get_feature_importance(self, feature_names=None):
        """Calculate feature importance based on gains"""
        if feature_names is None:
            feature_names = [f"Feature_{i}" for i in range(30)]
        
        importance = np.zeros(len(feature_names))
        
        def traverse(node, weight=1.0):
            if node['type'] == 'split':
                importance[node['feature_idx']] += node['gain'] * weight
                # Recursively traverse children with reduced weight
                left_weight = weight * (len(node['left']['class_distribution']) / node['n_samples'])
                right_weight = weight * (len(node['right']['class_distribution']) / node['n_samples'])
                traverse(node['left'], left_weight)
                traverse(node['right'], right_weight)
        
        traverse(self.tree)
        
        # Normalize
        importance = importance / np.sum(importance)
        
        # Create DataFrame
        feature_importance = pd.DataFrame({
            'feature': feature_names,
            'importance': importance
        }).sort_values('importance', ascending=False)
        
        return feature_importance

In [31]:
def stratified_split(X, y, test_size=0.2, random_state=42):
    """Stratified split for numpy arrays"""
    np.random.seed(random_state)
    
    # Convert to numpy if needed
    if hasattr(X, 'values'):
        X = X.values
    if hasattr(y, 'values'):
        y = y.values
    
    classes = np.unique(y)
    train_indices = []
    test_indices = []
    
    for cls in classes:
        cls_indices = np.where(y == cls)[0]
        np.random.shuffle(cls_indices)
        split_point = int(len(cls_indices) * (1 - test_size))
        train_indices.extend(cls_indices[:split_point])
        test_indices.extend(cls_indices[split_point:])
    
    np.random.shuffle(train_indices)
    np.random.shuffle(test_indices)
    
    X_train = X[train_indices]
    X_test = X[test_indices]
    y_train = y[train_indices]
    y_test = y[test_indices]
    
    return X_train, X_test, y_train, y_test

#### Correlation

In [45]:
# Get only numeric columns
numeric_df = df.select_dtypes(include=[np.number])

if target_col in numeric_df.columns:
    # Calculate correlation with target
    corr_with_target = numeric_df.corr()[target_col].drop(target_col).abs().sort_values(ascending=False)
    print("\nTop 10 features most correlated with target:")
    print(corr_with_target.head(15))


Top 10 features most correlated with target:
concave points_worst    0.793566
perimeter_worst         0.782914
concave points_mean     0.776614
radius_worst            0.776454
perimeter_mean          0.742636
area_worst              0.733825
radius_mean             0.730029
area_mean               0.708984
concavity_mean          0.696360
concavity_worst         0.659610
compactness_mean        0.596534
compactness_worst       0.590998
radius_se               0.567134
perimeter_se            0.556141
area_se                 0.548236
Name: target, dtype: float64


In [47]:
X = df[top_features.index.tolist()]
y = df[target_col]

In [32]:
print("\nSplitting data...")
X_train, X_test, y_train, y_test = stratified_split(X, y, test_size=0.2, random_state=42)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")
print(f"Train distribution: {np.bincount(y_train)}")
print(f"Test distribution: {np.bincount(y_test)}")



Splitting data...
Train size: 454
Test size: 115
Train distribution: [285 169]
Test distribution: [72 43]


In [51]:
# Train the decision tree
print("\n" + "="*60)
print("Training Decision Tree on Breast Cancer Dataset...")
print("="*60)

tree = SimpleDecisionTree(
    max_depth=5,  # Maximum depth of tree
    min_samples_split=10,  # Minimum samples to split a node
    min_impurity_decrease=0.02  # Minimum gain to split
)

tree.fit(X_train, y_train)


Training Decision Tree on Breast Cancer Dataset...


In [52]:
# Print the tree structure
print("\n" + "="*60)
print("Decision Tree Structure:")
print("="*60)
tree.print_tree(feature_names=X.columns.tolist())


Decision Tree Structure:
├── If perimeter_worst < 112.4347 (gain: 0.3326, samples: 454)
│   └── Then:
    ├── If concave points_worst < 0.1361 (gain: 0.0457, samples: 296)
    │   └── Then:
        └── Predict: Class 0 (samples: 265, distribution: {np.int64(0): 259, np.int64(1): 6})
    └── Else:
        ├── If concave points_worst < 0.1865 (gain: 0.1629, samples: 31)
        │   └── Then:
            ├── If area_worst < 728.3000 (gain: 0.1526, samples: 23)
            │   └── Then:
                ├── If compactness_worst < 0.2772 (gain: 0.0694, samples: 12)
                │   └── Then:
                    └── Predict: Class 0 (samples: 2, distribution: {np.int64(1): 1, np.int64(0): 1})
                └── Else:
                    └── Predict: Class 0 (samples: 10, distribution: {np.int64(0): 10})
            └── Else:
                ├── If perimeter_worst < 108.6000 (gain: 0.4628, samples: 11)
                │   └── Then:
                    └── Predict: Class 1 (samples: 7, dis

In [53]:
# Feature importance
print("\n" + "="*60)
print("Feature Importance:")
print("="*60)
feature_importance = tree.get_feature_importance(feature_names=X.columns.tolist())
print(feature_importance.head(10))


Feature Importance:
                 feature    importance
1        perimeter_worst  9.988596e-01
0   concave points_worst  1.104661e-03
5             area_worst  3.259766e-05
4         perimeter_mean  1.894800e-06
9        concavity_worst  1.243414e-06
11     compactness_worst  3.483003e-08
2    concave points_mean  0.000000e+00
6            radius_mean  0.000000e+00
3           radius_worst  0.000000e+00
8         concavity_mean  0.000000e+00


In [54]:
# Evaluate
print("\n" + "="*60)
print("Model Evaluation:")
print("="*60)

train_predictions = tree.predict(X_train)
test_predictions = tree.predict(X_test)

train_accuracy = np.mean(train_predictions == y_train)
test_accuracy = np.mean(test_predictions == y_test)

print(f"Training Accuracy: {train_accuracy:.2%}")
print(f"Test Accuracy: {test_accuracy:.2%}")


Model Evaluation:
Training Accuracy: 97.80%
Test Accuracy: 90.43%


In [55]:
# Classification report (without sklearn)
def classification_report_custom(y_true, y_pred):
    """Simple classification report"""
    classes = np.unique(y_true)
    print("\nClassification Report:")
    print("="*50)
    print(f"{'Class':<10} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
    print("-"*50)
    
    for cls in classes:
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        support = np.sum(y_true == cls)
        
        class_name = "Benign" if cls == 0 else "Malignant"
        print(f"{class_name:<10} {precision:.4f}     {recall:.4f}     {f1:.4f}     {support:<10}")
    
    # Overall accuracy
    accuracy = np.mean(y_true == y_pred)
    print("-"*50)
    print(f"\nOverall Accuracy: {accuracy:.2%}")

print("\nTest Set Performance:")
classification_report_custom(y_test, test_predictions)


Test Set Performance:

Classification Report:
Class      Precision    Recall       F1-Score     Support   
--------------------------------------------------
Benign     0.8861     0.9722     0.9272     72        
Malignant  0.9444     0.7907     0.8608     43        
--------------------------------------------------

Overall Accuracy: 90.43%


In [56]:
# Confusion matrix
print("\n" + "="*60)
print("Confusion Matrix:")
print("="*60)
cm = np.zeros((2, 2), dtype=int)
for i in range(len(y_test)):
    cm[y_test[i], test_predictions[i]] += 1

print(f"{'':<12} {'Predicted':>20}")
print(f"{'':<16} {'Benign':>10} {'Malignant':>10}")
print("-"*50)
print(f"{'Actual Benign':<12} {cm[0,0]:>10} {cm[0,1]:>10}")
print(f"{'Actual Malignant':<12} {cm[1,0]:>7} {cm[1,1]:>10}")
print("-"*50)


Confusion Matrix:
                        Predicted
                     Benign  Malignant
--------------------------------------------------
Actual Benign         70          2
Actual Malignant       9         34
--------------------------------------------------
